# Principles of Natural Language Processing Lab  
## Language Modelling: N-grams, N-gram Probabilities, Evaluation and Perplexity

**Course:** AME 5053 — Principles of Natural Language Processing Lab  
**Lab duration:** 3 hours  
**Mode:** Guided implementation + comparison + interpretation

### Learning outcomes
By the end of this lab, you should be able to:

1. construct unigram, bigram and trigram language models from a text corpus;
2. estimate N-gram probabilities using maximum-likelihood estimation (MLE);
3. compute the probability of a sentence under an N-gram model;
4. explain the effect of sentence-boundary symbols such as `<s>` and `</s>`;
5. evaluate a language model on unseen text using perplexity;
6. diagnose why an unsmoothed N-gram model can fail on unseen N-grams.

### What you must submit
Your notebook should contain:
- your predictions before running selected cells;
- completed implementations;
- at least one comparison between models;
- an error analysis using concrete examples;
- a short conclusion explaining what model behaviour you observed.

## 1. Corpus used in this lab

We will use a small, controlled corpus so that the counts can be inspected manually.

```text
students learn natural language processing
students learn machine learning
students study language models
language models predict words
language models assign probabilities
machine learning models learn patterns
natural language processing uses language models
```

The corpus is intentionally small. This makes the probability calculations transparent, but it will also expose an important limitation: **many valid word sequences will never occur in the training corpus**.

We will later split examples into **training** and **test** sentences so that perplexity is computed on data that the model did not simply memorize.

In [1]:
from collections import Counter, defaultdict
import math
import pandas as pd

raw_sentences = [
    "students learn natural language processing",
    "students learn machine learning",
    "students study language models",
    "language models predict words",
    "language models assign probabilities",
    "machine learning models learn patterns",
    "natural language processing uses language models",
]

raw_sentences

['students learn natural language processing',
 'students learn machine learning',
 'students study language models',
 'language models predict words',
 'language models assign probabilities',
 'machine learning models learn patterns',
 'natural language processing uses language models']

## 1.1 Tokenize the corpus
Run the helper below and inspect the output.

In [2]:
def tokenize(sentence):
    return sentence.lower().split()

tokenized = [tokenize(s) for s in raw_sentences]
tokenized

[['students', 'learn', 'natural', 'language', 'processing'],
 ['students', 'learn', 'machine', 'learning'],
 ['students', 'study', 'language', 'models'],
 ['language', 'models', 'predict', 'words'],
 ['language', 'models', 'assign', 'probabilities'],
 ['machine', 'learning', 'models', 'learn', 'patterns'],
 ['natural', 'language', 'processing', 'uses', 'language', 'models']]

## 2. Sentence boundaries

A language model should know not only which words can follow other words, but also where a sentence can begin and end.

For a bigram model we will represent:

```text
students learn machine learning
```

as

```text
<s> students learn machine learning </s>
```

For a trigram model, we need enough history at the beginning. We therefore use two start symbols:

```text
<s> <s> students learn machine learning </s>
```

This convention allows the model to estimate probabilities for sentence beginnings as well as sentence endings.

### Prediction 1 — before coding

Without running any code, answer:

1. Which word do you expect to be the most frequent unigram?
2. Which bigram do you expect to have a relatively high probability?
3. Do you expect a trigram model to assign non-zero probability to more or fewer unseen test sentences than a bigram model? Why?

Write your answers below.

**Your prediction:**  
- Most frequent unigram:  
- Likely high-probability bigram:  
- Bigram vs trigram on unseen text:

## 3. Build N-gram counts

For an N-gram model, we count neighbouring sequences of words.

For example, in:

```text
language models predict words
```

the bigrams are:

```text
(<s>, language)
(language, models)
(models, predict)
(predict, words)
(words, </s>)
```

The trigram model uses sequences of three tokens.

We will use these counts to estimate conditional probabilities.

In [16]:
def add_boundaries(tokens, n):
    # TODO:
    # Bigram model: add one <s> and one </s>
    if n==1:
      return list(tokens)

    # Trigram model: add two <s> symbols and one </s>
    return ['<s>']*(n-1) + list(tokens) + ['</s>']


print(add_boundaries(['a','b','c'],1))
print(add_boundaries(['a','b','c'],2))
print(add_boundaries(['a','b','c'],3))

['a', 'b', 'c']
['<s>', 'a', 'b', 'c', '</s>']
['<s>', '<s>', 'a', 'b', 'c', '</s>']


In [15]:
def make_ngrams(tokens,n):
  return [tuple(tokens[i:i+n]) for i in range (len(tokens)-n+1)]

print(make_ngrams(['a','b','c'],2))
print(make_ngrams(['a','b','c'],3))

[('a', 'b'), ('b', 'c')]
[('a', 'b', 'c')]


In [62]:
from collections import Counter

# Initialize Counters
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()
trigram_prefix_counts = Counter() # New counter for bigram prefixes in trigram context

for sentence_tokens in tokenized:
  unigram_counts.update(sentence_tokens)
  # For bigram model (P(w2|w1)), use n=2 boundaries for bigrams and their prefixes (w1)
  bigram_counts.update(make_ngrams(add_boundaries(sentence_tokens, 2),2))

  # For trigram model (P(w3|w1,w2)), use n=3 boundaries for trigrams and their prefixes (w1,w2)
  # First, get tokens with trigram boundaries
  trigram_bounded_tokens = add_boundaries(sentence_tokens, 3)
  trigram_counts.update(make_ngrams(trigram_bounded_tokens,3))
  # The context for trigrams are bigrams (w1,w2) from these same trigram-bounded tokens
  trigram_prefix_counts.update(make_ngrams(trigram_bounded_tokens,2))

print("unigram count: ",len(unigram_counts))
print("bigram count: ",len(bigram_counts))
print("trigram count: ",len(trigram_counts))
print("trigram_prefix_count: ",len(trigram_prefix_counts))

unigram count:  15
bigram count:  28
trigram count:  32
trigram_prefix_count:  29


### Inspect the most frequent N-grams
Create tables showing the 10 most frequent unigrams, bigrams and trigrams.

In [44]:
#10 most frequent unigrams, bigrams and trigrams
# print(unigram_counts.most_common(10))
# print(bigram_counts.most_common(10))
# print(trigram_counts.most_common(10))

#another way to display uni-grams,bi-grams and tri-grams
def top_table(counter,k=10):
  return pd.DataFrame(counter.most_common(k),columns=['ngram','count'])

display(top_table(unigram_counts))
display(top_table(bigram_counts))
display(top_table(trigram_counts))


,ngram,count
0,language,6
1,models,5
2,students,3
3,learn,3
4,natural,2
5,processing,2
6,machine,2
7,learning,2
8,study,1
9,predict,1


,ngram,count
0,"(language, models)",4
1,"(<s>, students)",3
2,"(students, learn)",2
3,"(natural, language)",2
4,"(language, processing)",2
5,"(machine, learning)",2
6,"(models, </s>)",2
7,"(<s>, language)",2
8,"(learn, natural)",1
9,"(processing, </s>)",1


,ngram,count
0,"(<s>, <s>, students)",3
1,"(<s>, students, learn)",2
2,"(natural, language, processing)",2
3,"(language, models, </s>)",2
4,"(<s>, <s>, language)",2
5,"(<s>, language, models)",2
6,"(students, learn, natural)",1
7,"(learn, natural, language)",1
8,"(language, processing, </s>)",1
9,"(students, learn, machine)",1


## 4. Maximum-likelihood estimation of N-gram probabilities

For a bigram model,

$$
P(w_i \mid w_{i-1})
=
\frac{C(w_{i-1}, w_i)}{C(w_{i-1})}
$$

For a trigram model,

$$
P(w_i \mid w_{i-2}, w_{i-1})
=
\frac{C(w_{i-2}, w_{i-1}, w_i)}
     {C(w_{i-2}, w_{i-1})}
$$

The key idea is simple: **among all occurrences of the history, how often was the next word the one we are asking about?**

This is a maximum-likelihood estimate because the probabilities are obtained directly from observed frequencies in the training corpus.

In [51]:
def bigram_probability(w1, w2, bigram_counts, unigram_context_counts):
    # TODO: implement P(w2 | w1)
    return bigram_counts[(w1,w2)]/unigram_context_counts[w1]

def trigram_probability(w1, w2, w3, trigram_counts, bigram_context_counts):
    # TODO: implement P(w3 | w1,w2)
    return trigram_counts[(w1,w2,w3)]/bigram_context_counts[(w1,w2)]

In [54]:
trigram_counts

Counter({('<s>', '<s>', 'students'): 3,
         ('<s>', 'students', 'learn'): 2,
         ('students', 'learn', 'natural'): 1,
         ('learn', 'natural', 'language'): 1,
         ('natural', 'language', 'processing'): 2,
         ('language', 'processing', '</s>'): 1,
         ('students', 'learn', 'machine'): 1,
         ('learn', 'machine', 'learning'): 1,
         ('machine', 'learning', '</s>'): 1,
         ('<s>', 'students', 'study'): 1,
         ('students', 'study', 'language'): 1,
         ('study', 'language', 'models'): 1,
         ('language', 'models', '</s>'): 2,
         ('<s>', '<s>', 'language'): 2,
         ('<s>', 'language', 'models'): 2,
         ('language', 'models', 'predict'): 1,
         ('models', 'predict', 'words'): 1,
         ('predict', 'words', '</s>'): 1,
         ('language', 'models', 'assign'): 1,
         ('models', 'assign', 'probabilities'): 1,
         ('assign', 'probabilities', '</s>'): 1,
         ('<s>', '<s>', 'machine'): 1,
         ('

### Probability checks
Compute and interpret the following:

- `P(learn | students)`
- `P(models | language)`
- `P(language | learn)`
- `P(models | study, language)`

In [58]:
bigram_prefix_counts = Counter()
for bigram_tuple, count in bigram_counts.items():
    bigram_prefix_counts[bigram_tuple[0]] += count

print(f"P(learn | students) = {bigram_probability('students','learn', bigram_counts, bigram_prefix_counts)}")
print(f"P(models | language) = {bigram_probability('language','models', bigram_counts, bigram_prefix_counts)}")
print(f"P(language | learn) = {bigram_probability('learn','language', bigram_counts, bigram_prefix_counts)}")

print(f"P(models | study, language) = {trigram_probability('study','language','models', trigram_counts, bigram_counts)}")

P(learn | students) = 0.6666666666666666
P(models | language) = 0.6666666666666666
P(language | learn) = 0.0
P(models | study, language) = 1.0


## 5. Sentence probability

Under a bigram model, the probability of a sentence is approximated using the Markov assumption:

$$
P(w_1,\ldots,w_m)
\approx
P(w_1\mid <s>)
\prod_{i=2}^{m}P(w_i\mid w_{i-1})
P(</s>\mid w_m)
$$

The trigram model conditions each word on the previous two tokens.

Because sentence probabilities are products of many values smaller than 1, they can become extremely small. In larger systems we therefore often work with **log probabilities**.

In [70]:
def bigram_sentence_probability(sentence, bigram_counts, context_counts):
    # 1. tokenize
    token = tokenize(sentence)
    # 2. add boundaries
    token = add_boundaries(token,2)
    # 3. multiply bigram probabilities
    prob = bigram_probability(token[0],token[1],bigram_counts,context_counts)
    for i in range(1,len(token)-1):
      prob *= bigram_probability(token[i],token[i+1],bigram_counts,context_counts)
    return prob

# Helper function to calculate sum of log probabilities for perplexity
def _bigram_sentence_log_probability(sentence, bigram_counts, context_counts):
    token = tokenize(sentence)
    token = add_boundaries(token,2)

    log_prob_sum = 0.0
    for i in range(len(token)-1):
        prob = bigram_probability(token[i], token[i+1], bigram_counts, context_counts)
        if prob == 0:
            return -math.inf # If any term is 0, log P is -inf, sum is -inf
        log_prob_sum += math.log(prob)
    return log_prob_sum

def trigram_sentence_probability(sentence, trigram_counts, context_counts):
    # 1. tokenize
    token = tokenize(sentence)
    # 2. add trigram boundaries
    token = add_boundaries(token,3)
    # 3. multiply trigram probabilities
    prob = trigram_probability(token[0],token[1],token[2],trigram_counts,context_counts)
    for i in range(2,len(token)-2):
      prob *= trigram_probability(token[i],token[i+1],token[i+2],trigram_counts,context_counts)
    return prob

# Helper function to calculate sum of log probabilities for perplexity
def _trigram_sentence_log_probability(sentence, trigram_counts, context_counts):
    token = tokenize(sentence)
    token = add_boundaries(token,3)

    log_prob_sum = 0.0
    for i in range(len(token)-2):
        prob = trigram_probability(token[i], token[i+1], token[i+2], trigram_counts, context_counts)
        if prob == 0:
            return -math.inf # If any term is 0, log P is -inf, sum is -inf
        log_prob_sum += math.log(prob)
    return log_prob_sum

### Compare sentence probabilities
Evaluate these two training-like sentences:

- `students learn machine learning`
- `language models assign probabilities`

Then explain why their probabilities differ.

In [66]:
# TODO: compute bigram and trigram sentence probabilities for the two sentences.
print(f"Bigram probability for 'students learn machine learning': {bigram_sentence_probability('students learn machine learning',bigram_counts,bigram_prefix_counts)}")
print(f"Bigram probability for 'language models assign probabilities': {bigram_sentence_probability('language models assign probabilities',bigram_counts,bigram_prefix_counts)}")

print(f"Trigram probability for 'students learn machine learning': {trigram_sentence_probability('students learn machine learning',trigram_counts,trigram_prefix_counts)}")
print(f"Trigram probability for 'language models assign probabilities': {trigram_sentence_probability('language models assign probabilities',trigram_counts,trigram_prefix_counts)}")

Bigram probability for 'students learn machine learning': 0.047619047619047616
Bigram probability for 'language models assign probabilities': 0.0380952380952381
Trigram probability for 'students learn machine learning': 0.10714285714285714
Trigram probability for 'language models assign probabilities': 0.07142857142857142


## 6. Evaluation on unseen text

A useful language model should perform well on text that was **not used to estimate its probabilities**.

We will use:

### Test sentence A
`students learn language models`

### Test sentence B
`language models learn patterns`

### Test sentence C
`students predict probabilities`

Before computing anything, inspect the corpus and predict which test sentences may contain unseen bigrams or trigrams.

### Prediction 2 — unseen sequences

Before running the next section, list at least one bigram/trigram that you think is unseen in each test sentence.

**A. students learn language models**  
Prediction:

**B. language models learn patterns**  
Prediction:

**C. students predict probabilities**  
Prediction:

## 7. Perplexity

Perplexity is a standard intrinsic evaluation measure for language models.

For a sequence containing \(N\) predicted tokens,

$$
PP(W)
=
P(W)^{-1/N}
$$

Equivalently, using log probabilities,

$$
PP(W)
=
\exp\left(
-\frac{1}{N}
\sum_{i=1}^{N}\log P(w_i \mid \text{history})
\right)
$$

Interpretation:

- **lower perplexity** means the model assigns higher probability to the observed test sequence;
- **higher perplexity** means the sequence is more surprising to the model;
- if an unsmoothed model encounters an N-gram with probability zero, the perplexity becomes infinite.

Perplexity should only be compared meaningfully when models are evaluated on the **same tokenization and same test data**.

In [73]:
def bigram_perplexity(sentence, bigram_counts, context_counts):
    # Calculate sum of log probabilities using the helper function
    log_prob_sum = _bigram_sentence_log_probability(sentence, bigram_counts, context_counts)

    if log_prob_sum == -math.inf:
        return math.inf

    # N is the number of predicted words, which is len(original_sentence_tokens) + 1 (for </s>)
    num_predicted_tokens = len(tokenize(sentence)) + 1

    # Perplexity formula: exp(- (1/N) * sum(log P))
    return math.exp(-log_prob_sum / num_predicted_tokens)

def trigram_perplexity(sentence, trigram_counts, context_counts):
    # Calculate sum of log probabilities using the helper function
    log_prob_sum = _trigram_sentence_log_probability(sentence, trigram_counts, context_counts)

    if log_prob_sum == -math.inf:
        return math.inf

    # N is the number of predicted words, which is len(tokenize(sentence)) + 1 (for </s>)
    num_predicted_tokens = len(tokenize(sentence)) + 1

    # Perplexity formula: exp(- (1/N) * sum(log P))
    return math.exp(-log_prob_sum / num_predicted_tokens)

In [77]:
test_sentences = [
    "students learn language models",
    "language models learn patterns",
    "students predict probabilities",
]

# Collect data for DataFrame
results = []

for sentence in test_sentences:
    bg_pp = bigram_perplexity(sentence, bigram_counts, bigram_prefix_counts)
    tg_pp = trigram_perplexity(sentence, trigram_counts, trigram_prefix_counts)
    results.append({"sentence": sentence, "bigram_perplexity": bg_pp, "trigram_perplexity": tg_pp})

# Create DataFrame from collected results
data = pd.DataFrame(results)

display(data)

,sentence,bigram_perplexity,trigram_perplexity
0,students learn language models,inf,inf
1,language models learn patterns,2.394694,inf
2,students predict probabilities,inf,inf


## 8. Diagnose the model
For every test sentence with infinite perplexity, print the first unseen N-gram that causes the failure.

In [76]:
def first_unseen_ngram(sentence, n, ngram_counts):
    tokens = tokenize(sentence)
    bounded_tokens = add_boundaries(tokens, n)
    ngrams = make_ngrams(bounded_tokens, n)

    for ngram in ngrams:
        if ngram_counts[ngram] == 0:
            return ngram
    return None

print("Diagnosing test sentences for unseen N-grams:")
for sentence in test_sentences:
    print(f"\nSentence: '{sentence}'")
    unseen_bigram = first_unseen_ngram(sentence, 2, bigram_counts)
    if unseen_bigram:
        print(f"  First unseen bigram: {unseen_bigram}")
    else:
        print(f"  All bigrams seen.")

    unseen_trigram = first_unseen_ngram(sentence, 3, trigram_counts)
    if unseen_trigram:
        print(f"  First unseen trigram: {unseen_trigram}")
    else:
        print(f"  All trigrams seen.")

Diagnosing test sentences for unseen N-grams:

Sentence: 'students learn language models'
  First unseen bigram: ('learn', 'language')
  First unseen trigram: ('students', 'learn', 'language')

Sentence: 'language models learn patterns'
  All bigrams seen.
  First unseen trigram: ('language', 'models', 'learn')

Sentence: 'students predict probabilities'
  First unseen bigram: ('students', 'predict')
  First unseen trigram: ('<s>', 'students', 'predict')


## 9. Interpretation and error analysis

Answer in complete sentences.

1. Which model gave lower perplexity on the test sentences that both models could score?
2. Did the higher-order model always perform better? Explain using the size of this corpus.
3. Identify one unseen bigram and one unseen trigram from the test set.
4. Why does a zero probability create a serious problem when sentence probability is computed by multiplication?
5. What would you expect to happen if the training corpus became much larger?

## 10. Viva / discussion questions

1. Why is an N-gram language model called a probabilistic model?
2. What is the Markov assumption in a bigram model?
3. Why are sentence boundary symbols useful?
4. Why can a trigram model be more data-hungry than a bigram model?
5. Why is perplexity usually preferred over raw sentence probability when comparing sequences of different lengths?
6. Why can an unsmoothed N-gram model assign infinite perplexity to a perfectly grammatical sentence?